[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AGRIdigital-Climbat/PlantillaMapa/blob/main/02_Precipitacion/01_precipitacion.ipynb)

# Precipitacion: regimen, estacionalidad y meses criticos
## PROSUCO - CLIMBAT | Analisis por municipio de Bolivia (GEE + Colab)

> **Area de estudio = tu GeoJSON.** Este notebook usa por defecto:
>
> `RUTA_AOI = "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"` (compartido `Cualquiera con el enlace`).
> El GeoJSON **siempre manda**: al cargarse, `roi`, `ZONA['bbox']`, `ZONA['centro']`
> y `area_km2` se recalculan desde el poligono. El `bbox` de CONFIG solo es respaldo
> si cambias a `AOI_MODE = "bbox"`.

**Orden de ejecucion:** 1) Instalacion -> 2) Autenticacion GEE (`EE_PROJECT`) ->
3) CONFIG (anos + satelite + umbrales) -> 4) AOI desde Drive -> 5) celdas del modulo.
Solo edita la celda **CONFIGURACION MAESTRA**.


In [12]:
# ============================================================
# 1.1 Instalación de librerías (ejecutar una sola vez en Colab)
# ============================================================

import importlib
import subprocess
import sys


def _asegurar(paquete, modulo=None):
    """Instala el paquete solo si no está disponible."""
    try:
        importlib.import_module(modulo or paquete)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", paquete]
        )


_asegurar("earthengine-api", "ee")
_asegurar("geemap")
_asegurar("geopandas")
_asegurar("pandas")
_asegurar("numpy")
_asegurar("matplotlib")

import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

print("Librerías listas.")
print("earthengine-api:", ee.__version__)
print("geemap:", geemap.__version__)

Librerías listas.
earthengine-api: 1.7.39
geemap: 0.38.8


In [13]:
# ============================================================
# 1.2 Autenticación en Google Earth Engine
# ============================================================
# >>> REEMPLAZAR por el id del proyecto de Cloud de PROSUCO/CLIMBAT <<<

EE_PROJECT = "gen-lang-client-0636031685"

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

print("Earth Engine inicializado correctamente.")
print("Proyecto:", EE_PROJECT)

Earth Engine inicializado correctamente.
Proyecto: gen-lang-client-0636031685


## Configuración maestra

> **✏️ Esta es la única celda que se edita para diagnosticar una zona nueva.**

| Bloque | Qué define | Verificación previa |
|---|---|---|
| `ZONA` | nombre, bbox `[oeste, sur, este, norte]` y punto de vista | dibujar en [geojson.io](https://geojson.io) o en el editor GEE y copiar coordenadas |
| `AOI_MODE`/`RUTA_AOI` | fuente del AOI: `"bbox"` o `"auto"` (polígono **GeoJSON**) | exportar el polígono desde QGIS como **GeoJSON**; en `RUTA_AOI` se puede pegar un **enlace de Google Drive** (el notebook lo descarga solo) o una ruta de Drive |
| `PUNTOS_LOCALES` | manantiales/bofedales del saber local | levantamiento con las comunidades |
| `FECHA_INICIO / FECHA_FIN` | ventana analítica (**30+ años**) | default 1996-01-01 → 2026-12-31 |
| `CALENDARIO` | meses de siembra/cosecha y estación seca | observadores Yapuchiri |
| `UMBRALES` | todos los cortes analíticos en un solo lugar | criterio técnico + experiencia local |
| `ESCALAS` | resolución de cómputo (m); subir si la zona es grande | — |

In [14]:
# ===========================================================================
# CONFIGURACIÓN MAESTRA — ÚNICA CELDA QUE SE EDITA PARA CADA ZONA
# ===========================================================================

# --- 1. Zona de intervención ------------------------------------------------
# Fuente del AOI:
#   "bbox" -> usa el rectángulo ZONA["bbox"] de abajo (rápido, sin Drive)
#   "auto" -> usa un polígono .geojson. RUTA_AOI admite 3 formas:
#       * Enlace o ID de Google Drive  -> se DESCARGA solo (no subes nada a mano)
#           "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"
#           "11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N"
#       * Ruta en tu Drive ya montado  -> se lee directo, SIN copiar a /content
#           "/content/drive/MyDrive/CLIMBAT/zona.geojson"
#       * Carpeta en Drive             -> toma el único .geojson que contenga
#           "/content/drive/MyDrive/CLIMBAT"
AOI_MODE = "auto"
RUTA_AOI = "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"

ZONA = {
    # Coordenadas geocodificadas: Escoma (La Paz, Bolivia) ~3832 msnm
    # (solo se usan como respaldo cuando AOI_MODE = "bbox")
    "nombre": "Escoma, La Paz (prueba)",
    "bbox": [-69.33, -15.86, -68.93, -15.46],   # [oeste, sur, este, norte]
    "centro": [-69.1289, -15.6615],             # punto de vista del mapa
    "zoom": 10,
}

# --- 2. Puntos levantados con comunidades (saber local) ---------------------
# Listas de [longitud, latitud]. Dejar vacías si aún no hay levantamiento.
PUNTOS_LOCALES = {
    "manantiales": [],
    "bofedales": [],
}

# --- 3. Ventana analítica: 30+ años hacia atrás desde el año actual ---------
# Regla de la plantilla: 1996-01-01 -> 2026-12-31 (todo análisis de rango de años usa esta).
# Los datasets usan su período nativo si es mayor: CHIRPS 1981+ · ERA5-Land 1950+
# · JRC 1984–2021 · MODIS 2000+ (NDVI) / 2001+ (MCD12Q1 agricultura) · SMAP 2015+
FECHA_INICIO = "1996-01-01"
FECHA_FIN = "2026-12-31"

# --- 4. Calendario agrícola local (verificar con observadores Yapuchiri) ----
CALENDARIO = {
    "siembra": 11,       # mes de siembra (1–12)
    "cosecha": 4,        # mes de cosecha (1–12)
    "meses_estacion_seca": [5, 6, 7, 8, 9, 10],   # se usa en M10 (bofedales)
}

# --- 5. Umbrales analíticos (ajustar con criterio técnico y local) ----------
UMBRALES = {
    "helada_c": 0.0,               # °C mínima para contar un día de helada (M4)
    "helada_grave_c": -2.0,        # °C de helada grave (daño severo) (M4)
    "mes_seco_mm": 30.0,           # mm/mes para clasificar mes seco (M2)
    "spi_severo": -1.5,            # umbral de sequía severa SPI-3 (M5)
    "ndvi_anomalia_z": -1.5,       # anomalía NDVI severa (M7)
    "manantial_occurrencia": 100,  # % ocurrencia JRC mínima (M9)
    "manantial_drenaje_max": 50,   # celdas acumuladas máx. (cabeceras) (M9)
    "bofedal_altitud_min": 3000,   # msnm mínimo (M10)
    "bofedal_altitud_max": 5200,   # msnm máximo (M10)
    "bofedal_pendiente_max": 5.0,  # grados de pendiente máxima (M10)
    "bofedal_seasonality": 8,      # meses con agua JRC mínimos (M10)
    "bofedal_ndvi_seco": 0.25,     # NDVI mínimo en estación seca (M10)
    "clases_agricolas": [12, 14],  # LC_Type1 MCD12Q1: 12=cultivos, 14=mosaico (M11)
}

# --- 6. Escalas de cómputo (metros) -----------------------------------------
# Subir si la zona es muy grande (más rápido, menos detalle)
ESCALAS = {"clima": 5000, "modis": 250, "jrc": 90, "detalle": 30}
TILE_SCALE = 4

# ===========================================================================
# A partir de aquí no se edita nada
# ===========================================================================

MESES = list(range(1, 13))
MESES_ES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
            "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

roi = None   # se construye en la celda 1.4 (bbox o GeoJSON según AOI_MODE)
centro = None
ANIO_INI = int(FECHA_INICIO[:4])
ANIO_FIN = int(FECHA_FIN[:4])

# Años COMPLETOS (se excluye el año en curso de métricas y mapas anuales)
ANIOS_COMPLETOS = list(range(ANIO_INI, ANIO_FIN))
FILTRO_ANIOS_COMPLETOS = ee.Filter.calendarRange(
    ANIOS_COMPLETOS[0], ANIOS_COMPLETOS[-1], "year"
)

print("Zona:", ZONA["nombre"])
print("Ventana analítica:", FECHA_INICIO, "->", FECHA_FIN,
      f"({ANIO_FIN - ANIO_INI + 1} años calendario)")
print("Años completos para métricas:", ANIOS_COMPLETOS[0], "-", ANIOS_COMPLETOS[-1])
print("Calendario: siembra mes", CALENDARIO["siembra"],
      "| cosecha mes", CALENDARIO["cosecha"])
print("Umbrales activos:", UMBRALES)

Zona: Escoma, La Paz (prueba)
Ventana analítica: 1996-01-01 -> 2026-12-31 (31 años calendario)
Años completos para métricas: 1996 - 2025
Calendario: siembra mes 11 | cosecha mes 4
Umbrales activos: {'helada_c': 0.0, 'helada_grave_c': -2.0, 'mes_seco_mm': 30.0, 'spi_severo': -1.5, 'ndvi_anomalia_z': -1.5, 'manantial_occurrencia': 100, 'manantial_drenaje_max': 50, 'bofedal_altitud_min': 3000, 'bofedal_altitud_max': 5200, 'bofedal_pendiente_max': 5.0, 'bofedal_seasonality': 8, 'bofedal_ndvi_seco': 0.25, 'clases_agricolas': [12, 14]}


In [15]:
# ============================================================
# 1.3 Utilidades comunes (reutilizadas por todos los módulos)
# ============================================================

# Parámetros de reducción seguros contra "User memory limit exceeded":
#   bestEffort=True -> GEE sube la escala automáticamente si no cabe en memoria.
#   tileScale=8     -> divide el cálculo en tiles más pequeños (menos memoria c/u).
REDUCIR = dict(maxPixels=1e9, tileScale=8, bestEffort=True)
REDUCIR_AREA = dict(maxPixels=1e10, tileScale=16, bestEffort=True)


def apilar(imagenes, nombres):
    """Une imágenes de una banda en una sola imagen multibanda."""
    pila = None
    for img, nombre in zip(imagenes, nombres):
        banda = ee.Image(img).select(0).rename(nombre)
        pila = banda if pila is None else pila.addBands(banda)
    return pila


def reducir_bandas(imagenes, nombres, escala, tam=6, reductor=None, params=None):
    """reduceRegion de muchas bandas EN BLOQUES (una petición por bloque).

    Evita el error de memoria que ocurre al reducir de una sola vez una imagen
    con decenas de bandas (cada banda puede depender de cientos/miles de imágenes
    diarias). Devuelve un dict {nombre_banda: valor}.
    """
    reductor = reductor or ee.Reducer.mean()
    params = params or REDUCIR
    salida = {}
    for i in range(0, len(imagenes), tam):
        pila = apilar(imagenes[i:i + tam], nombres[i:i + tam])
        salida.update(
            pila.reduceRegion(reductor, roi, escala, **params).getInfo()
        )
    return salida


def composite_mensual(coleccion, anios, banda=None):
    """Reduce una colección diaria a UN compuesto por año-mes (una imagen c/u).

    Aligera ~30x la memoria de las climatologías de 30 años. `anios` debe listar
    solo años con dato (evita colecciones vacías). Con `banda`, deja una sola banda.
    """
    imgs = []
    for y in anios:
        for m in MESES:
            i0 = f"{y}-{m:02d}-01"
            i1 = (pd.Timestamp(i0) + pd.offsets.MonthBegin(1)).strftime("%Y-%m-%d")
            sub = coleccion.filterDate(i0, i1)
            if banda:
                sub = sub.select(banda)
            imgs.append(sub.mean().set("system:time_start", ee.Date(i0).millis()))
    return ee.ImageCollection(imgs)


def borde_aoi(color="FF0000", ancho=2):
    """Capa de SOLO el borde del AOI (relleno transparente).

    Evita que el relleno de color tape el mapa y meta ruido en la interpretación;
    deja el límite del área bien visible por encima de las demás capas.
    """
    return roi.style(color=color, fillColor="00000000", width=ancho)


def promediar_aoi(imagen, banda, escala):
    """Promedio de una banda dentro del AOI -> valor Python (o None)."""
    d = (
        imagen.select(banda)
        .reduceRegion(ee.Reducer.mean(), roi, escala, **REDUCIR)
        .getInfo()
    )
    return d.get(banda)


def serie_aoi(coleccion, banda, escala, bloque_anios=5):
    """Extrae el promedio del AOI por imagen, EN BLOQUES de años.

    Procesar la serie completa de una vez agota la memoria; por eso se trocea
    (por defecto cada 5 años) y se concatenan los resultados.
    """
    registros = []

    def _una_imagen(img):
        st = (
            img.select(banda)
            .reduceRegion(ee.Reducer.mean(), roi, escala, **REDUCIR)
        )
        return ee.Feature(None, {
            "date": ee.Date(img.get("system:time_start")).format("yyyy-MM-dd"),
            "valor": st.get(banda),
        })

    for y0 in range(ANIO_INI, ANIO_FIN + 1, bloque_anios):
        y1 = min(y0 + bloque_anios - 1, ANIO_FIN)
        sub = coleccion.filterDate(f"{y0}-01-01", f"{y1 + 1}-01-01")
        fc = ee.FeatureCollection(sub.map(_una_imagen)).getInfo()
        registros += [f["properties"] for f in fc["features"]]
    return registros


print("Utilidades: apilar(), reducir_bandas(), composite_mensual(), "
      "promediar_aoi(), serie_aoi()")

Utilidades: apilar(), reducir_bandas(), composite_mensual(), promediar_aoi(), serie_aoi()


In [16]:
# ============================================================
# 1.4 Área de estudio (AOI) y mapa base
# ============================================================
# Fuente del AOI (ver CONFIG):
#   AOI_MODE = "bbox" -> rectángulo ZONA["bbox"].
#   AOI_MODE = "auto" -> polígono desde un archivo .geojson en Google Drive.
#                        (En QGIS: Exportar -> Guardar entidades como -> GeoJSON,
#                         CRS EPSG:4326; subir a Drive y ajustar RUTA_AOI.)

def borde_aoi(color="FF0000", ancho=2):
    """Capa de SOLO el borde del AOI (relleno transparente)."""
    # ee.Geometry NO tiene .style() en la API de Python -> se envuelve en FeatureCollection
    fc = ee.FeatureCollection([ee.Feature(roi)])
    return fc.style(color=color, fillColor="00000000", width=ancho)


if AOI_MODE == "auto":
    from google.colab import drive
    drive.mount("/content/drive")

    import os
    import re
    import glob
    import geopandas as gpd

    ruta = str(RUTA_AOI).strip()

    # (a) Enlace o ID de Google Drive -> descarga automática (sin subir nada a mano).
    #     El archivo debe estar compartido como "Cualquiera con el enlace".
    if ruta.startswith("http") or ("/" not in ruta and len(ruta) >= 20):
        _asegurar("gdown")
        import gdown

        m = re.search(r"(?:/d/|id=)([-\w]{20,})", ruta)
        fid = m.group(1) if m else ruta
        ext = (
            ".zip" if ruta.lower().endswith(".zip") else
            ".kml" if ruta.lower().endswith(".kml") else
            ".gpkg" if ruta.lower().endswith(".gpkg") else
            ".geojson"
        )
        destino = "/content/zona_aoi" + ext
        print("Descargando AOI desde Google Drive...")
        gdown.download(id=fid, output=destino, quiet=False)
        ruta = destino
    # (b) Carpeta en Drive -> usar el único .geojson que contenga
    elif os.path.isdir(ruta):
        cands = sorted(glob.glob(os.path.join(ruta, "*.geojson")))
        if not cands:
            raise FileNotFoundError(f"No se encontró ningún .geojson en {ruta}")
        ruta = cands[0]
    # (c) Ruta de archivo en el Drive montado -> se lee directo (sin copiar)

    gdf = gpd.read_file(ruta).to_crs(4326)
    try:
        geometria = gdf.geometry.union_all()      # shapely >= 2 (disuelve features)
    except AttributeError:
        geometria = gdf.geometry.unary_union      # shapely < 2
    gdf = gpd.GeoDataFrame(geometry=[geometria], crs=4326)

    roi = geemap.geopandas_to_ee(gdf).geometry()

    minx, miny, maxx, maxy = geometria.bounds
    ZONA["bbox"] = [minx, miny, maxx, maxy]
    ZONA["centro"] = [(minx + maxx) / 2, (miny + maxy) / 2]
    ZONA["zoom"] = 10
    print("AOI cargado desde:", ruta)
else:
    roi = ee.Geometry.Rectangle(ZONA["bbox"], proj="EPSG:4326", geodesic=False)

centro = ee.Geometry.Point(ZONA["centro"])
area_km2 = roi.area(maxError=1e6).divide(1e6).getInfo()

Map_aoi = geemap.Map()
Map_aoi.centerObject(roi, ZONA["zoom"])
Map_aoi.add_basemap("HYBRID")
Map_aoi.addLayer(borde_aoi(), {}, f"AOI: {ZONA['nombre']}")

# Puntos del saber local (si se cargaron en CONFIG)
for tipo, color in (("manantiales", "cyan"), ("bofedales", "magenta")):
    coords = PUNTOS_LOCALES.get(tipo, [])
    if coords:
        Map_aoi.addLayer(
            ee.Geometry.MultiPoint(coords),
            {"color": color},
            f"{tipo.capitalize()} (saber local)",
        )

print(f"Zona: {ZONA['nombre']}")
print(f"Área del AOI: {area_km2:,.1f} km²")
print(f"Extensión (bbox): {ZONA['bbox']}")
print("Puntos de saber local cargados:",
      {k: len(v) for k, v in PUNTOS_LOCALES.items()})

Map_aoi

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Descargando AOI desde Google Drive...


Downloading...
From: https://drive.google.com/uc?id=11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N
To: /content/zona_aoi.geojson
100%|██████████| 6.32k/6.32k [00:00<00:00, 19.0MB/s]


AOI cargado desde: /content/zona_aoi.geojson
Zona: Escoma, La Paz (prueba)
Área del AOI: 5,638.9 km²
Extensión (bbox): [-68.088142, -19.88671, -67.170426, -19.050138]
Puntos de saber local cargados: {'manantiales': 0, 'bofedales': 0}


Map(center=[-19.436538457151894, -67.61413826547428], controls=(WidgetControl(options=['position', 'transparen…

# M2. Precipitación: régimen, estacionalidad y meses críticos
**Dimensiones:** *estacionalidad* (meses secos/húmedos), *ocurrencia* (% de días lluviosos),
*frecuencia* de años secos (complementada en M5), *tendencia* (serie anual).

* **Primario:** `UCSB-CHG/CHIRPS/DAILY` (0,05°, 1981–actual): el estándar para precipitación
  histórica en zonas con pocas estaciones, validado en los Andes. **Se usa la ventana analítica
  CONFIG (1996–2026)** por la regla de 30 años; para usar el registro completo de CHIRPS
  descomentar la opción 1981 en M2.1.
* **Alternativas:** IMERG (satélite, 2000+, resolución horaria — bueno para eventos cortos),
  ERA5-Land (reanálisis 1950+, 11 km), TerraClimate (mensual 4,6 km).
* **Qué se calcula:** climatología mensual de 12 mapas, % de días lluviosos (≥ 1 mm),
  concentración en los 3 meses más lluviosos (índice simple de estacionalidad) y serie anual.

> Requiere haber ejecutado la celda de CONFIGURACIÓN y las utilidades de la Parte I.

In [17]:
# ============================================================
# M2.1b Precipitación: (izq) mensual por año · (der) anual por año
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import ee

# --- Definición de variables ausentes de M2.1 ---
precip = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")

# Generar compuestos mensuales (colección temporal acumulada mensual)
precip_mensual = composite_mensual(precip, ANIOS_COMPLETOS, "precipitation")

# Climatología de 12 meses media (df_mes)
imgs_clim = []
for m in MESES:
    sub_m = precip_mensual.filter(ee.Filter.eq("month", m))
    imgs_clim.append(sub_m.mean().rename(f"m_{m}"))

val_clim = reducir_bandas(imgs_clim, [f"m_{m}" for m in MESES], ESCALAS["clima"], tam=12)
df_mes = pd.DataFrame({
    "mes": MESES,
    "mm": [val_clim.get(f"m_{m}") or 0 for m in MESES]
})

# --- Datos: mm de cada mes en cada año (serie_aoi trocea en bloques de 5 años)
registros = serie_aoi(precip_mensual, "precipitation", ESCALAS["clima"])
df_serie = (pd.DataFrame(registros)
            .assign(fecha=lambda d: pd.to_datetime(d["date"]))
            .dropna(subset=["valor"]))
df_serie["mes"] = df_serie["fecha"].dt.month      # columnas explícitas: pivot_table
df_serie["anio"] = df_serie["fecha"].dt.year      # no admite rutas tipo "fecha.dt.x"

df_anios = (df_serie.pivot_table(index="mes", columns="anio", values="valor")
            .reindex(MESES))          # filas = meses 1..12, columnas = años

tot_anio = df_anios.sum()             # mm acumulados de cada año
anio_seco, anio_humedo = tot_anio.idxmin(), tot_anio.idxmax()
media_anual = tot_anio.mean()

# --- Figura: dos paneles lado a lado -----------------------------------------
fig, ejes = plt.subplots(1, 2, figsize=(17, 5))

# Izquierda: UNA LÍNEA POR AÑO (eje X meses, eje Y mm)
ax = ejes[0]
for anio in df_anios.columns:
    serie = df_anios[anio]
    if anio == anio_seco:
        ax.plot(MESES, serie, lw=2.2, color="#ef6c00", label=f"{anio} (más seco)")
    elif anio == anio_humedo:
        ax.plot(MESES, serie, lw=2.2, color="#1565c0", label=f"{anio} (más húmedo)")
    else:
        ax.plot(MESES, serie, lw=1, color="#90a4ae", alpha=0.5)
ax.plot(MESES, df_mes["mm"], lw=3, color="#00695c",
        label=f"Media {ANIOS_COMPLETOS[0]}–{ANIOS_COMPLETOS[-1]}")
ax.axhline(UMBRALES["mes_seco_mm"], color="#d32f2f", ls="--", lw=1,
           label=f"Mes seco (< {UMBRALES['mes_seco_mm']:.0f} mm)")
ax.set_xticks(MESES)
ax.set_xticklabels(MESES_ES)
ax.set_xlabel("Mes")
ax.set_ylabel("Precipitación mensual (mm)")
ax.set_title(f"1) Mensual por año ({df_anios.shape[1]} líneas)")
ax.set_ylim(0, None)
ax.grid(alpha=0.3)
ax.legend(fontsize=8, ncol=2)

# Derecha: BARRAS de precipitación anual por año
ax2 = ejes[1]
bajo_media = tot_anio < media_anual
colores = np.where(bajo_media, "#ef6c00", "#1976d2")
ax2.bar(tot_anio.index, tot_anio.values, color=colores)

# Línea SOLO de la media (sin línea "seco" que chocaba con las barras)
ax2.axhline(media_anual, color="#00695c", lw=2, ls="--")

# Dato de seco/húmedo como anotación sobre SU barra (no como línea global)
for anio, color in ((anio_seco, "#e65100"), (anio_humedo, "#0d47a1")):
    alto = tot_anio[anio]
    ax2.annotate(f"{anio}\n{alto:,.0f} mm",
                 xy=(anio, alto), xytext=(0, 6), textcoords="offset points",
                 ha="center", va="bottom", fontsize=8, color=color, weight="bold")

ax2.set_xlabel("Año")
ax2.set_ylabel("Precipitación anual (mm)")
ax2.set_title("2) Total anual por año")
ax2.grid(alpha=0.3, axis="y")
ax2.tick_params(axis="x", rotation=45)

# Leyenda hecha a mano: CADA elemento tiene su entrada
ax2.legend(handles=[
    Patch(facecolor="#1976d2", edgecolor="none",
          label="Año sobre la media"),
    Patch(facecolor="#ef6c00", edgecolor="none",
          label="Año bajo la media"),
    plt.Line2D([], [], color="#00695c", lw=2, ls="--",
               label=f"Media {media_anual:,.0f} mm"),
], fontsize=9, loc="upper right")

plt.tight_layout()
plt.show()

print(f"Años trazados: {df_anios.columns.min()}–{df_anios.columns.max()} "
      f"({df_anios.shape[1]})")
print(f"Media anual: {media_anual:,.0f} mm | seco: {anio_seco} "
      f"({tot_anio[anio_seco]:,.0f} mm) | húmedo: {anio_humedo} "
      f"({tot_anio[anio_humedo]:,.0f} mm)")
print(f"Años bajo la media: {int(bajo_media.sum())} de {len(tot_anio)}")

EEException: Image.select: Invalid band number (0) specified to select. Input only contains 0 bands.

In [ ]:
# ============================================================
# M2.2 Serie anual, años extremos y mapa con selector de años
# ============================================================

import ipywidgets as widgets
from IPython.display import display as ipy_display

anios_clim = list(ANIOS_COMPLETOS)
print(f"Ventana anual: {anios_clim[0]}-{anios_clim[-1]} ({len(anios_clim)} años completos)")

anuales_m2 = [
    precip.filterDate(f"{y}-01-01", f"{y + 1}-01-01").sum() for y in anios_clim
]
val_anual = reducir_bandas(
    anuales_m2, [f"p_{y}" for y in anios_clim], ESCALAS["clima"], tam=6,
)

df_precip = pd.DataFrame({
    "anio": anios_clim,
    "mm": [val_anual.get(f"p_{y}") for y in anios_clim],
})
media = df_precip["mm"].mean()
p33 = df_precip["mm"].quantile(0.33)
df_precip["seco"] = df_precip["mm"] < p33

display(df_precip.round(0))

print(f"Media: {media:,.0f} mm | P33 (umbral de año seco local): {p33:,.0f} mm")
print("Frecuencia de años secos:",
      int(df_precip["seco"].sum()), "de", len(df_precip))
print("Año más seco:", int(df_precip.loc[df_precip["mm"].idxmin(), "anio"]),
      f"({df_precip['mm'].min():,.0f} mm)")
print("Año más húmedo:", int(df_precip.loc[df_precip["mm"].idxmax(), "anio"]),
      f"({df_precip['mm'].max():,.0f} mm)")

# --- Gráfico -----------------------------------------------------------------
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(df_precip["anio"], df_precip["mm"], color="#90a4ae", lw=1, zorder=1)
ax.scatter(df_precip["anio"], df_precip["mm"],
           c=np.where(df_precip["seco"], "#d32f2f", "#1976d2"), zorder=3)
ax.axhline(media, color="gray", ls="--", label=f"Media {media:,.0f} mm")
ax.axhline(p33, color="#d32f2f", ls=":", label=f"P33 {p33:,.0f} mm (año seco)")
ax.set_title("Precipitación anual (puntos rojos = años secos)")
ax.set_ylabel("mm/año")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# --- Colección anual con system:time_start ----------------------------------
def img_anual(y):
    img = (precip
           .filterDate(f"{y}-01-01", f"{y + 1}-01-01")
           .sum()
           .rename("p")
           .clip(roi))
    return img.set("system:time_start", ee.Date(f"{y}-01-01").millis()).set("year", y)

imgs_anuales = {y: img_anual(y) for y in anios_clim}

col_anual_m2 = ee.ImageCollection(list(imgs_anuales.values())).sort("system:time_start")
precip_media = col_anual_m2.reduce(ee.Reducer.mean()).rename("mm_anio").clip(roi)

# --- Escalado dinámico de colores según el AOI -------------------------------
stats_p = precip_media.reduceRegion(
    reducer=ee.Reducer.percentile([2, 98]),
    geometry=roi,
    scale=ESCALAS["clima"],
    maxPixels=1e13,
    bestEffort=True,
)
mm_min = stats_p.get("mm_anio_p2")
mm_max = stats_p.get("mm_anio_p98")
min_val = mm_min.getInfo() if mm_min else 0
max_val = mm_max.getInfo() if mm_max else 1500
if min_val is None:
    min_val = 0
if max_val is None or max_val <= min_val:
    max_val = min_val + 1000
print(f"Rango de visualización (P2–P98): {min_val:.0f} – {max_val:.0f} mm/año")

vis_p = {
    "min": min_val,
    "max": max_val,
    "palette": ["ffffd9", "c7e9b4", "41b6c4", "2c7fb8", "253494"],
}

# --- Mapa --------------------------------------------------------------------
Map_m2 = geemap.Map()
Map_m2.centerObject(roi, ZONA["zoom"])
Map_m2.add_basemap("HYBRID")

# Capa de la media (apagada por defecto)
Map_m2.addLayer(precip_media, vis_p, "Precipitación media anual (mm)", shown=False)

# AOI encima
Map_m2.addLayer(borde_aoi(), {}, "AOI")

# Barra de color a topright (lejos de los controles de zoom)
Map_m2.add_colorbar(vis_p, label="mm/año",
                    orientation="horizontal", position="topright",
                    layer_name="Precipitación anual (mm)")

# --- Capa inicial: último año ------------------------------------------------
anio_default = anios_clim[-1]
NOMBRE_CAPA = "Precipitación anual (mm)"
Map_m2.addLayer(imgs_anuales[anio_default], vis_p, NOMBRE_CAPA, shown=True)

# --- Selector desplegable de años --------------------------------------------
def actualizar_anio(change):
    y = change["new"]
    # Eliminar la capa previa con el mismo nombre
    for capa in list(Map_m2.layers):
        if capa.name == NOMBRE_CAPA:
            Map_m2.remove_layer(capa)
    # Añadir la capa del año seleccionado
    Map_m2.addLayer(imgs_anuales[y], vis_p, NOMBRE_CAPA, shown=True)

selector = widgets.Dropdown(
    options=anios_clim,
    value=anio_default,
    description="Año:",
    layout=widgets.Layout(width="200px"),
    style={"description_width": "50px"},
)
selector.observe(actualizar_anio, names="value")

# Caja con etiqueta y selector, separada del mapa
caja_selector = widgets.HBox(
    [
        widgets.HTML("<b>Selecciona el año:</b>"),
        selector,
    ],
    layout=widgets.Layout(
        padding="10px",
        border="1px solid #ccc",
        width="fit-content",
        margin="8px 0 0 0",
    ),
)

# Selector ANTES del mapa
ipy_display(caja_selector)
ipy_display(Map_m2)

## Interpretación de M2
* **Estacionalidad:** qué tan concentrada está la lluvia (p. ej. > 70% en 3 meses = régimen muy
  estacional, típico del Altiplano: verano húmedo e invierno seco). Define la ventana de siembra
  y la necesidad de riego/cosecha de agua.
* **Ocurrencia (% días lluviosos):** días con ≥ 1 mm; mide la *probabilidad de lluvia* en cada
  mes, más informativo que la media sola.
* **Frecuencia de años secos:** cuántos años cayeron por debajo de su propio P33 (umbral local).
  En M5 se profundiza con SPI.
* **Tendencia:** observa la serie anual; si hay años recientes sistemáticamente bajos, amerita
  pruebas formales (Mann-Kendall) antes de afirmar cambio climático.

**Preguntas guía**
1. ¿Los meses que ustedes llaman "de sequía" coinciden con los meses secos del mapa?
2. ¿El calendario de siembra (mes CONFIG) cabe dentro de la temporada húmeda?
3. ¿Qué años recuerdan como críticos? ¿aparecen en la serie?

## Cierre: checklist del modulo

- [ ] Parte base ejecutada (instalacion, autenticacion GEE, CONFIG, AOI del GeoJSON).
- [ ] Mapa(s) revisados contra el borde rojo del AOI (el GeoJSON siempre manda).
- [ ] Serie/tabla leidas + interpretacion contrastada con saber local (Yapuchiri/PACHASOL).
- [ ] Periodo y dataset anotados junto a cada numero (p. ej. `CHIRPS 1981-2025`).

### Como cambiar de satelite o de anos (patron en todos los modulos)

```python
FECHA_INICIO = "1996-01-01"  # en la celda CONFIG: ventana analitica
FECHA_FIN = "2026-12-31"
# --- Primario (default) ---
precip = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
# --- Alternativas: comentar la primaria y descomentar la elegida ---
# precip = ee.ImageCollection("NASA/GPM_L3/IMERG_V06")
# Cada alternativa indica resolucion, periodo y para que sirve.
```
